## Overview

This notebook is intended to:
* Demonstrate how to programmatically use tracing functionalities available in TGI.

The five types of tracing this notebook covers are:
1. **Source**: trace upstream of a specified grid element. 
2. **All Sources**: trace all producers in the circuit of the specified grid element. Note that `Source` trace is a subset of `All Sources` trace.
3. **Down**: trace downstream of a specified grid element to all other grid elements.
4. **Connected**: trace all grid elements in the same circuit as the specified grid element. 
5. **Same Voltage**: trace all grid elements in the circuit with the same voltage as the specified grid element.

Please refer to the [main_concepts.ipynb](../2_main_concepts/main_concepts.ipynb) notebook for a high-level introduction to and simpler examples of the core views and functions available in Awesense's Energy Data Model (EDM).

---

## Set up

In [1]:
import getpass
import pandas as pd
from shapely import wkb
import plotly.express as px
import urllib.parse

pd.set_option('display.max_columns', None)

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [2]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

%load_ext sql
%sql postgresql://$edm_name:$edm_password@$edm_address/edm
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False

# Delete the credential variables for security purpose.
del edm_name, edm_password


EDM login information


**Custom Functions**

In [3]:
def query_to_df(result):
    """
    Transform sql query result to dataframe, and
    expand out `meta` column into separate columns.
    """
    
    # Turn a query output into a Python dataframe.
    df = result.DataFrame()

    # Separate the `meta` column currently saved as JSONB into individual columns.
    # Replace NaN values to empty string for ease of spotting non-NaN values.
    df = pd.concat([df.drop(['meta'], axis=1),
                    df['meta'].apply(pd.Series).fillna('')], axis=1)
    
    # Transform the hex string form of geometry to coordinates.
    df['geometry'] = df['geometry'].apply(lambda x: wkb.loads(x, hex=True)) 
    
    return df


def plot_pie(df, interest_colname, title):
    """
    Compute counts of each value in `interest_colname` column,  
    plot a pie chart of the breakdown, and 
    return the counts dataframe.
    """
    
    # Create a dataframe of counts of each value.
    df_cts = df[interest_colname].value_counts().to_frame().reset_index()
    
    # Create a pie plot.
    fig = px.pie(df_cts, names=interest_colname, values='count',
                title=title)
    
    # Show the count values, percentages and labels for the pie plot.
    fig.update_traces(textposition='inside', textinfo='percent+label+value')
    fig.show()
    
    # Return the count dataframe sorted alphabetically by in the input column. 
    return df_cts.sort_values(interest_colname) 


def phase_table(df):
    """
    Return a table of phase breakdowns by grid element type.
    """

    # Create a dataframe of counts by both types and phases.
    df_counts = df[['type','phases']].value_counts().to_frame().reset_index()
    
    # Rename the first column as 'count'.
    df_counts.rename(columns={0:'count'}, inplace=True)
    
    # Pivot the dataframe to have phases as columns.
    df_phases = df_counts.pivot_table(index='type', columns='phases', fill_value=0)

    # Return the count dataframe sorted alphabetically by `type`.
    return df_phases.sort_values('type') 

---

## Example

### Input Parameters

We'll take a look at the `awefice` grid. In the database, this property is referred to as a `grid_id`.

In [4]:
grid_id = 'awefice'

Check when this grid was last updated. 

In [5]:
%%sql

SELECT last_updated
FROM grid
WHERE grid_id = '{grid_id}';

last_updated
2021-01-01 20:00:00+00:00


The below are different element types available in the `awefice` grid.

In [6]:
%%sql

SELECT DISTINCT type as element_type
FROM grid_element
WHERE grid_id = '{grid_id}'
ORDER BY element_type;

element_type
ACLineSegment
Battery
Busbar
Cabinet
CircuitBreaker
Disconnector
EVCharger
Fuse
Jumper
Meter


Enter one of the element types from the above output to explore further in the `grid_element` view.

In [7]:
element_type = input('Enter the element type of interest: ')

The below are all available grid element ID for the specified grid `type`.

In [8]:
%%sql

SELECT grid_element_id
FROM grid_element
WHERE grid_id = '{grid_id}'
    AND type = '{element_type}'
ORDER BY grid_element_id;

grid_element_id
m_1
m_10
m_11
m_12
m_13
m_14
m_15
m_16
m_17
m_18


Choose one of the grid element ID from the above output to conduct tracing on.

In [9]:
grid_element_id = input('Enter the grid element id of interest: ')

### Trace

Choose one of the tracing options: `Source`, `All Sources`, `Down`, `Connected`, and `Same Voltage`.

In [10]:
trace_option = input('Enter the tracing option: ')

Call an appropriate SQL query based on the specified tracing option.

In [11]:
if trace_option.lower() == 'source':
    
    # Set the third argument to `true` to retrieve the top feeder.
    result = %sql SELECT * \
            FROM grid_get_sources('{grid_id}', '{grid_element_id}', 'true');   
    
elif trace_option.lower() == 'all sources':
    
    # Set the third argument to `false` to retrieve all feeders.
    result = %sql SELECT * \
            FROM grid_get_sources('{grid_id}', '{grid_element_id}', 'false');
    
elif trace_option.lower() == 'down':
    
    # Set the third argument to `false` to exclude itself from the output.
    result = %sql SELECT * \
            FROM grid_get_downstream('{grid_id}', '{grid_element_id}', 'false');
    
elif trace_option.lower() == 'connected':
    
    result = %sql SELECT * \
            FROM grid_get_connected('{grid_id}', '{grid_element_id}');
    
elif trace_option.lower() == 'same voltage':
    
    result = %sql SELECT * \
            FROM grid_get_same_voltage('{grid_id}', '{grid_element_id}');

else:
    print('Invalid input for tracing option. Please enter one of the following: Source, All Sources, Down, Connected.')

In [12]:
# Turn the sql output to a Python dataframe.
df = query_to_df(result)

# Return the first few rows.
df.head()

,grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,name,label,system,rating_a,materials,latitude_1,latitude_2,make_model,line_length,longitude_1,longitude_2,is_insulated,ph_ph_voltage,voltage_level,wire_diameter,hvmv_parent_element,is_operation_status,mvlv_parent_element,is_ground_deployable,sdp_id,address,interval,latitude,estimated,longitude,lv_feeder,mv_feeder,read_type,tariff_id,meter_type,is_critical,meter_number,zone_level_1,zone_level_2,zone_level_3,zone_level_4,energy_source,maximal_demand,commission_date,meter_multiplier,source_rating_kw,type_of_consumer,connected_to_lvmv,replacement_cycle,service_drop_type,standard_profile_id,maximal_demand_units,parent_transformer_id,business_identifier_id,feeder_name,enclosure_id,control_method,conducting_status,normal_operation_status,switchable_element_type,load_loss,ownership,rating_kva,no_load_loss,primary_voltage,maintenance_type,manufacture_date,installation_type,secondary_voltage,winding_configuration
0,awefice,line_segment_10,ACLineSegment,line_segment,ABC,False,False,False,False,None,con_9,con_21,both,None,"LINESTRING (-123.10668133248 49.2675566618634,...",10,10,delta,184.0,Al_Steel,,,#2 ACSR,18.99,,,False,12500.0,MV,8.25,,True,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,awefice,line_segment_12,ACLineSegment,line_segment,ABC,False,False,False,False,None,con_344,con_346,both,None,LINESTRING (-123.106653447079 49.2673852487066...,12,12,delta,184.0,Al_Steel,,,#2 ACSR,72.6,,,False,12500.0,MV,8.25,,True,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,awefice,line_segment_134,ACLineSegment,line_segment,ABC,False,False,False,False,None,con_21,con_342,both,None,LINESTRING (-123.106687800683 49.2673858056582...,134,134,delta,184.0,Al_Steel,,,#2 ACSR,2.0,,,False,12500.0,MV,8.25,,True,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,awefice,line_segment_18,ACLineSegment,line_segment,A,False,False,False,False,None,con_34,con_44,1to2,transformer_16,LINESTRING (-123.105656937445 49.2673758369888...,18,18,,150.0,Al,,,2 AWG Triplex,11.99,,,True,480.0,LV,6.54,,True,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,awefice,line_segment_3,ACLineSegment,line_segment,ABC,False,False,False,False,None,con_3,con_5,both,None,LINESTRING (-123.107134801681 49.2675640126912...,3,3,delta,184.0,Al_Steel,,,#2 ACSR,9.01,,,False,12500.0,MV,8.25,,True,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


The above dataframe can be processed further as needed. For example, TGI only returns a top feeder for the `Source` option whereas the notebook here returns all grid_elements. Applying the `is_producer==True` filter on the `df` will mirror the same functionality as in TGI.

In [13]:
# Filter the dataframe to rows with is_producer=True.
df[df['is_producer']==True]

,grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,name,label,system,rating_a,materials,latitude_1,latitude_2,make_model,line_length,longitude_1,longitude_2,is_insulated,ph_ph_voltage,voltage_level,wire_diameter,hvmv_parent_element,is_operation_status,mvlv_parent_element,is_ground_deployable,sdp_id,address,interval,latitude,estimated,longitude,lv_feeder,mv_feeder,read_type,tariff_id,meter_type,is_critical,meter_number,zone_level_1,zone_level_2,zone_level_3,zone_level_4,energy_source,maximal_demand,commission_date,meter_multiplier,source_rating_kw,type_of_consumer,connected_to_lvmv,replacement_cycle,service_drop_type,standard_profile_id,maximal_demand_units,parent_transformer_id,business_identifier_id,feeder_name,enclosure_id,control_method,conducting_status,normal_operation_status,switchable_element_type,load_loss,ownership,rating_kva,no_load_loss,primary_voltage,maintenance_type,manufacture_date,installation_type,secondary_voltage,winding_configuration
11,awefice,transformer_2,Transformer,transformer,ABC,False,True,False,False,None,con_1,con_3,both,None,POINT (-123.107138 49.26756406),tr_2,tr_2,,,,,,,,,,,,HV/MV,,,True,,,,,,,,,,,,,,,,,,,,,,1989-01-01,,,,,,,,,,,,enc_1,,,,,246932.0,utility,20000.0,24775.0,200000.0,oil,,station,12500.0,


For the demonstration of this notebook, we will continue using the full dataframe with unfiltered data.

In [14]:
# Create a dataframe of counts of each value.
df_cts = df['type'].value_counts().to_frame().reset_index()
df_cts

,type,count
0,ACLineSegment,6
1,CircuitBreaker,2
2,Transformer,2
3,Meter,1
4,Fuse,1


In [15]:
# Create a pie plot for the breakdown of grid element types.
df_types = plot_pie(df, 'type', 
                     'Breakdown of Grid Element Types for ' + trace_option.title() + ' Trace')

In [16]:
# Return a table of grid element type counts.
df_types

,type,count
0,ACLineSegment,6
1,CircuitBreaker,2
4,Fuse,1
3,Meter,1
2,Transformer,2


In [17]:
# Create a pie plot for the breakdown of phases.
df_phases = plot_pie(df, 'phases', 
                     'Breakdown of Phases for ' + trace_option.title() + ' Trace')

In [18]:
# Return a table of phase counts.
df_phases

,phases,count
1,A,4
0,ABC,8


In [19]:
# Create a table of phase counts by grid element types.
phase_table(df)

count     
phases             A  ABC
type                     
ACLineSegment    1.0  5.0
CircuitBreaker   0.0  2.0
Fuse             1.0  0.0
Meter            1.0  0.0
Transformer      1.0  1.0

### Combining Trace Functions

Combining multiple trace functions provides the flexibility to find an answer to additional questions.

For example, a grid element's parent transformer can be identified by: 
1. Returning a list of transformers in the upstream using `grid_get_sources()`.
2. Entering the list of transformers from step 1 to `grid_get_same_voltage()` to get all grid elements with the same voltage as a given transformer.
3. Matching the input grid element to the returned grid elements from step 2 and filter to the respective transformer as the final output.

In [20]:
grid_element_id = input('Enter the grid element id of interest: ')

In [21]:
%%sql

SELECT ggs.grid_element_id AS transformer
FROM grid_get_sources('{grid_id}', '{grid_element_id}', true) ggs
LEFT JOIN grid_get_same_voltage('{grid_id}', ggs.grid_element_id) ggsv
    ON true
WHERE ggs.type = 'Transformer'
    AND ggsv.grid_element_id = '{grid_element_id}';


transformer
transformer_47


---